# Auswertung der Experimente (Datenexploration)
Dieses Notebook lädt die generierte CSV-Datei (`experiment_ergebnisse.csv`) und visualisiert die Ergebnisse für die Bachelorarbeit. 
Es generiert automatisch Publikations-bereite Graphen (mit 300 DPI für den Export) und berechnet die statistische Signifikanz.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

# Professionelles Design für die Bachelorarbeit (wie in wissenschaftlichen Papern)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.dpi': 300, 'font.size': 12}) # 300 DPI für scharfen Export

csv_filename = "experiment_ergebnisse.csv"

if not os.path.exists(csv_filename):
    print(f"FEHLER: Die Datei {csv_filename} wurde nicht gefunden.")
else:
    # WICHTIG: Das Trennzeichen ist ein Semikolon, wie im Log-Skript definiert!
    df = pd.read_csv(csv_filename, sep=";")
    print(f"Daten erfolgreich geladen! Anzahl der durchgeführten Experimente: {len(df)}")
    display(df.head()) # Zeigt die ersten 5 Zeilen an

In [ ]:
# Filtern auf das Haupt-Setup (50 Samples, 450 Unknowns)
df_main = df[(df["Samples pro Zielklasse"] == 50) & (df["Anzahl Unknown Samples"] == 450)].copy()

if len(df_main) > 0:
    print(f"Anzahl der Durchläufe (Seeds) für das Haupt-Setup: {len(df_main)}")
    
    # Daten für Seaborn "schmelzen" (umstrukturieren) für den direkten Vergleich
    metrics_to_plot = ["Base_Accuracy", "Strat_Accuracy", 
                       "Base_F1_Score", "Strat_F1_Score", 
                       "Base_AUC_ROC", "Strat_AUC_ROC"]
    
    df_melted = df_main.melt(id_vars=["Seed"], value_vars=metrics_to_plot, 
                             var_name="Metrik_Modell", value_name="Score (%)")
    
    # Aufspalten in Metrik-Name und Modell-Name
    df_melted["Modell"] = df_melted["Metrik_Modell"].apply(lambda x: "Baseline" if "Base" in x else "Strategie")
    df_melted["Metrik"] = df_melted["Metrik_Modell"].apply(lambda x: x.split("_", 1)[1])
    
    # Plot generieren
    plt.figure(figsize=(10, 6))
    ax = sns.boxplot(data=df_melted, x="Metrik", y="Score (%)", hue="Modell", width=0.5, fliersize=5)
    
    # Auch die einzelnen Datenpunkte (Seeds) als Punkte einzeichnen
    sns.stripplot(data=df_melted, x="Metrik", y="Score (%)", hue="Modell", 
                  dodge=True, color='black', alpha=0.6, ax=ax, legend=False)
    
    plt.title("Leistungsvergleich: Baseline vs. Strategie (N=50 pro Klasse)")
    plt.ylim(0, 100) # Prozentskala
    plt.tight_layout()
    plt.savefig("plot_haupt_metriken.png") # Speichert das Bild für deine Arbeit
    plt.show()
else:
    print("Noch keine Daten für das Setup N=50 und Unknown=450 vorhanden.")

In [ ]:
# Prüfen, ob wir überhaupt verschiedene Datenmengen getestet haben
if df["Samples pro Zielklasse"].nunique() > 1:
    plt.figure(figsize=(8, 5))
    
    # Liniendiagramme
    sns.lineplot(data=df, x="Samples pro Zielklasse", y="Base_Accuracy", label="Baseline (Standard Transfer)", marker="o")
    sns.lineplot(data=df, x="Samples pro Zielklasse", y="Strat_Accuracy", label="Strategie (OOD-Augmentation)", marker="s")
    
    plt.title("Lernkurve: Einfluss der gelabelten Datenmenge")
    plt.xlabel("Gelabelte Bilder pro Zielklasse")
    plt.ylabel("Test Accuracy (%)")
    plt.legend()
    plt.tight_layout()
    plt.savefig("plot_learning_curve.png")
    plt.show()
else:
    print("Hinweis: Es wurde bisher nur mit einer Datenmenge trainiert. Für eine Lernkurve musst du das Experiment mit anderen Werten für 'NUM_SAMPLES_PER_CLASS' wiederholen.")

In [ ]:
# Prüfen, ob verschiedene Unknown-Mengen getestet wurden
if df["Anzahl Unknown Samples"].nunique() > 1:
    # Wir filtern auf N=50, falls sich das überlappt
    df_ablation = df[df["Samples pro Zielklasse"] == 50]
    
    plt.figure(figsize=(8, 5))
    sns.lineplot(data=df_ablation, x="Anzahl Unknown Samples", y="Strat_Accuracy", marker="o", color="green")
    
    # Die Baseline ist als horizontale Linie (Referenz) einzuzeichnen
    base_mean = df_ablation["Base_Accuracy"].mean()
    plt.axhline(base_mean, color="red", linestyle="--", label=f"Baseline (Mean: {base_mean:.1f}%)")
    
    plt.title("Ablationsstudie: Einfluss der 'Unknown'-Datenmenge")
    plt.xlabel("Anzahl der OOD-Bilder (BloodMNIST)")
    plt.ylabel("Strategie Test Accuracy (%)")
    plt.legend()
    plt.tight_layout()
    plt.savefig("plot_ablation.png")
    plt.show()
else:
    print("Hinweis: Für die Ablationsstudie musst du das Skript mit verschiedenen Werten für 'NUM_UNKNOWN_SAMPLES' ausführen.")

In [ ]:
if len(df_main) >= 3: # Wir brauchen mind. 3 Seeds für einen t-Test
    print("==========================================")
    print("STATISTISCHE AUSWERTUNG (Für die Bachelorarbeit)")
    print("==========================================\n")
    
    strat_acc = df_main["Strat_Accuracy"]
    base_acc = df_main["Base_Accuracy"]
    
    # Mittelwerte und Standardabweichungen
    mean_strat, std_strat = strat_acc.mean(), strat_acc.std()
    mean_base, std_base = base_acc.mean(), base_acc.std()
    
    # Gepaarter t-Test
    # Wir nutzen "rel" (related/paired), da beide Modelle auf denselben Daten-Seeds trainiert wurden
    t_stat, p_val = stats.ttest_rel(strat_acc, base_acc)
    
    print(f"Baseline Accuracy:  {mean_base:.2f}% (± {std_base:.2f}%)")
    print(f"Strategie Accuracy: {mean_strat:.2f}% (± {std_strat:.2f}%)\n")
    print(f"Durchschnittliche Verbesserung: +{(mean_strat - mean_base):.2f} Prozentpunkte\n")
    
    print("--- Signifikanz-Test (Gepaarter t-Test) ---")
    print(f"p-Wert: {p_val:.5f}")
    
    if p_val < 0.05:
        print("\nFazit: Die Strategie ist STATISTISCH SIGNIFIKANT besser (p < 0.05).")
        print("Du kannst in der Arbeit schreiben: 'Die OOD-Augmentationsstrategie führte zu einer signifikanten Verbesserung der Klassifikationsleistung (p < 0.05).'")
    else:
        print("\nFazit: Der Unterschied ist statistisch NICHT signifikant (p >= 0.05).")
        print("Mögliche Ursache: Zu wenige Seeds (N) oder die Streuung ist zu hoch.")
else:
    print("Führe das Haupt-Experiment (50 Samples, 450 Unknowns) mit mindestens 3 verschiedenen Seeds durch, um einen statistischen T-Test berechnen zu können!")